# LUMA - lokalny urok Mistrzostwa Artystycznego
LUMA to lokalny jarmark, który rośnie z tygodnia na tydzień: coraz więcej wystawców, więcej gośc (miejscowych i turystów), okazjonalne promocje i sezonowość.

# Import bibliotek

In [ ]:
import numpy as np
import pandas as pd
from itertools import chain
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# Generowanie syntetycznych danych

Ustawienie ziarna RNG

In [ ]:
RNG_SEED = 20251021
rng = np.random.default_rng(RNG_SEED)
print("Ustawiono ziarno RNG: ", RNG_SEED)

# Zdefiniowanie "stałych"

In [ ]:
LUMA_FULL = "Lokalny Urok Mistrzostwa Artystycznego"
CATEGORIES = ["Ceramika", "Tekstylia", "Biżuteria", "Rękodzieło", "Przetwory", "Dekoracje", "Stolarka", "Masarstwo", "Alkohole"]

# Przygotowanie bazowych danych syntetycznych

Katalog kategorii i ich bazowe parametry generatywne(syntetyczne)

In [ ]:
catalog = pd.DataFrame({
    "category": CATEGORIES,
    "demand_loc_base": [120, 180, 160, 80, 160, 150, 70, 90, 110], # średni popyt lokalny (szt./tydz.)
    "demand_tur_base": [60, 40, 80, 30, 90, 30, 50, 40, 80], # średni popyt turystyczny (szt./tydz.)
    "price_pln_base": [8, 120, 150, 230, 40, 35, 250, 215, 135], # średnia cena jednostkowa (PLN/szt.)
})

Wyliczony "typowe" przychód tygodniowy (lokalni + turyści) * cena

In [ ]:
catalog["revenue_avg_pln"] = (catalog["demand_loc_base"] + catalog["demand_tur_base"]) * catalog["price_pln_base"]

In [ ]:
catalog

# Generowanie popytu lokalnego i turystycznego na podstawie danych bazowych z losowym szumem(multiplikatywnym) i opcjami boosta

In [ ]:
def determine_demand(catalog, rng, variance=0.15, boost_loc=0.0, boost_tur=0.0):
    df = catalog[["category", "demand_loc_base", "demand_tur_base"]].copy()
    eps_loc = np.clip(rng.normal(loc=1.0, scale=variance, size=len(df)), 0.5, 1.8)
    eps_tur = np.clip(rng.normal(loc=1.0, scale=variance, size=len(df)), 0.5, 1.8)
    df["demand_loc"] = (df["demand_loc_base"] * (1 + boost_loc)) * eps_loc
    df["demand_tur"] = (df["demand_tur_base"] * (1 + boost_tur)) * eps_tur 
    df["demand_loc"] = np.maximum(df["demand_loc"].round().astype(int), 0)
    df["demand_tur"] = np.maximum(df["demand_tur"].round().astype(int), 0)
    return df[["category", "demand_loc", "demand_tur"]]

# Wyliczanie przychodu lokalnego, turystycznego i łącznego

In [ ]:
def count_revenue(catalog, demand_df):
    df = demand_df.merge(catalog[["category", "price_pln_base"]], on="category", how="left")
    df["revenue_loc"] = df["demand_loc"] * df["price_pln_base"]
    df["revenue_tur"] = df["demand_tur"] * df["price_pln_base"]
    df["revenue_pln"] = df["revenue_loc"] + df["revenue_tur"]
    return df[["category", "revenue_loc", "revenue_tur", "revenue_pln"]]

# Generowanie danych tygodniowych

In [ ]:
def generate_weekly_data(catalog, rng, variance=0.15, boost_loc=0.0, boost_tur=0.0):
    demand_df = determine_demand(catalog, rng, variance=variance, boost_loc=boost_loc, boost_tur=boost_tur)
    revenue_df = count_revenue(catalog, demand_df)
    return revenue_df

In [ ]:
t1 = generate_weekly_data(catalog, rng)
t1 = t1.merge(catalog[["category", "price_pln_base"]], on="category", how="left")
print("Tydzień 1 - przychody per kategoria (PLN)")
display(t1)
print("Łączny przychód w tygodniu 1 (PLN): ", t1["revenue_pln"].sum())

In [ ]:
t1.plot.bar()

In [ ]:
t1[["revenue_loc", "revenue_tur"]].plot.bar()

In [ ]:
t1[["revenue_loc", "revenue_tur"]].plot.barh(stacked=True)

In [ ]:
heights = t1["revenue_pln"]
plt.bar(range(len(heights)), heights)

In [ ]:
heights = t1["revenue_pln"]
categories = t1["category"]

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(categories, heights)
plt.show()

In [ ]:
plt.figure(figsize=(12, 4))
plt.bar(categories, heights)
plt.title("LUMA - Przychody per kategoria w tygodniu 1")
plt.xlabel("Kategoria")
plt.ylabel("Przychód (PLN)")
plt.show()

# Etykieta wartości i formatowanie liczb

In [ ]:
def format_pln(x: float) -> str:
    return f"{int(round(x)):,} zł".replace(",", "\u202f")

In [ ]:
plt.figure(figsize=(12, 4))
bars = plt.bar(categories, heights)
ymin, ymax = plt.ylim() # pobranie wartości min i max dla y
pad = (ymax - ymin) * 0.07
plt.ylim(ymin, ymax + pad) # ustawienie nowej wartości y max
for bar in bars:
    x = bar.get_x() + bar.get_width() / 2
    y = bar.get_height()
    plt.text(x, y, format_pln(y), ha="center", va="bottom")
plt.title("LUMA - Przychody per kategoria w tygodniu 1")
plt.xlabel("Kategoria")
plt.ylabel("Przychód (PLN)")
plt.tight_layout()
plt.show()

In [ ]:
def pad_axis(factor=0.07):
    ymin, ymax = plt.ylim() # pobranie wartości min i max dla y
    pad = (ymax - ymin) * factor
    plt.ylim(ymin, ymax + pad) # ustawienie nowej wartości y max

In [ ]:
def label_bars_pln(bars):
    for bar in bars:
        x = bar.get_x() + bar.get_width() / 2
        y = bar.get_height()
        plt.text(x, y, format_pln(y), ha="center", va="bottom")

# Formatowanie etykiet podziałki osi

In [ ]:
def format_tys(x: float, decimals: int = 0) -> str:
    v = x / 1000.0
    if decimals == 0:
        s = f"{int(round(v)):,}".replace(",", "\u202f")
    else:
        s = f"{v:,.{decimals}f}".replace(",", "\u202f").replace(".", ",")
    return s + " tys."

In [ ]:
def format_mln(x: float, decimals: int = 0) -> str:
    v = x / 1000000.0
    if decimals == 0:
        s = f"{int(round(v)):,}".replace(",", "\u202f")
    else:
        s = f"{v:,.{decimals}f}".replace(",", "\u202f").replace(".", ",")
    return s + " mln."

In [ ]:
plt.figure(figsize=(12, 4))
bars = plt.bar(categories, heights)
pad_axis()
label_bars_pln(bars)
ypos, ylab_old = plt.yticks()
ylab_new = [format_tys(y) for y in ypos]
plt.yticks(ypos, ylab_new)
plt.title("LUMA - Przychody per kategoria w tygodniu 1")
plt.xlabel("Kategoria")
plt.ylabel("Przychód (PLN)")
plt.tight_layout()
plt.show()

In [ ]:
def format_ytics_tys():
    ypos, _ = plt.yticks()
    ylab_new = [format_tys(y) for y in ypos]
    plt.yticks(ypos, ylab_new)

# Sortowanie kategorii dla łatwiejszego porównania

In [ ]:
t1_sorted = t1.sort_values("revenue_pln", ascending=False).reset_index()
categories = t1_sorted["category"]
heights = t1_sorted["revenue_pln"]

In [ ]:
plt.figure(figsize=(12, 4))
bars = plt.bar(categories, heights)
pad_axis()
label_bars_pln(bars)
format_ytics_tys()
plt.title("LUMA - Przychody per kategoria w tygodniu 1")
plt.xlabel("Kategoria")
plt.ylabel("Przychód (PLN)")
plt.tight_layout()
plt.show()

# Siatka w tle wykresu

In [ ]:
plt.figure(figsize=(12, 4))
bars = plt.bar(categories, heights, color="darkseagreen")
pad_axis()
label_bars_pln(bars)
format_ytics_tys()
plt.grid(axis="y", linestyle=":", alpha=0.6)
plt.title("LUMA - Przychody per kategoria w tygodniu 1")
plt.xlabel("Kategoria")
plt.ylabel("Przychód (PLN)")
plt.tight_layout()
plt.show()

# Analiza i wizualizacja danych - 2 serie

Mini-fest przyciąga klientów: lokalni +13%, turyści +20%

In [ ]:
boost_loc = 0.13
boost_tur = 0.20

t2 = generate_weekly_data(catalog, rng, boost_loc=boost_loc, boost_tur=boost_tur)
t2 = t2.merge(catalog[["category", "price_pln_base"]], on="category", how="left")
print("Tydzień 2 - przychody per kategoria (PLN)")
display(t2)
print("Łączny przychód w tygodniu 1: ", format_pln(t1["revenue_pln"].sum()))
print("Łączny przychód w tygodniu 2: ", format_pln(t2["revenue_pln"].sum()))

przygotowanie danych do wykresów

In [ ]:
categories = t1_sorted["category"]
t1_revenue_total = t1_sorted["revenue_pln"]
t2_revenue_total = t2.set_index("category").loc[categories]["revenue_pln"]
x_locs = np.arange(len(categories))
bar_width = 0.4
bar_half = bar_width / 2

In [ ]:
plt.figure(figsize=(13,5))
plt.bar(x_locs-bar_half, t1_revenue_total, width=bar_width, 
        label="Tydzień 1", tick_label=categories)
plt.bar(x_locs+bar_half, t2_revenue_total, width=bar_width, 
        label="Tydzień 2")
plt.legend()
plt.tight_layout()
plt.show()

Łączenie operowania po seriach

In [ ]:
plt.figure(figsize=(13,5))
bars_1 = plt.bar(x_locs-bar_half, t1_revenue_total, width=bar_width, label="Tydzień 1")
bars_2 = plt.bar(x_locs+bar_half, t2_revenue_total, width=bar_width, label="Tydzień 2")
plt.xticks(x_locs, categories)
pad_axis()
label_bars_pln(chain(bars_1, bars_2))
format_ytics_tys()

plt.legend()
plt.tight_layout()
plt.show()

aktualizacja wyświetlania etykiet wartości

In [ ]:
def format_sep(x: float) -> str:
    return f"{int(round(x)):,}".replace(",", "\u202f")

In [ ]:
plt.figure(figsize=(13,5))
bars_1 = plt.bar(x_locs-bar_half, t1_revenue_total, width=bar_width, label="Tydzień 1", color="forestgreen")
bars_2 = plt.bar(x_locs+bar_half, t2_revenue_total, width=bar_width, label="Tydzień 2", color="steelblue")
plt.xticks(x_locs, categories)
pad_axis()
for b in bars_1:
    x, y = b.get_x(), b.get_height()
    plt.text(x, y, format_sep(y), ha="left", va="bottom")
for b in bars_2:
    x, y = b.get_x() + b.get_width(), b.get_height()
    plt.text(x, y, format_sep(y), ha="right", va="bottom")
format_ytics_tys()
plt.grid(axis="y", linestyle=":", alpha=0.6)
plt.title("LUMA - Porównanie przychodów per kategoria w tygodniu 1 i 2")
plt.xlabel("Kategoria")
plt.ylabel("Przychód (PLN)")
plt.legend()
plt.tight_layout()
plt.show()

# Analiza i wizualizacja danych - prezentacja zmian bezwzglednych

przygotowanie rónic i procentowych zmian między tygodniami 1 i 2

In [ ]:
order = CATEGORIES
t1_revenue_total = t1.set_index("category").loc[order]["revenue_pln"]
t2_revenue_total = t2.set_index("category").loc[order]["revenue_pln"]

base = t1_revenue_total
delta = t2_revenue_total - t1_revenue_total

pct = np.divide(
    delta,
    base,
    out=np.full_like(base, np.nan, dtype=float), 
    where=base != 0 
) * 100

t_delta = pd.DataFrame({
    "category": order,
    "t1_pln": t1_revenue_total.astype(int),
    "t2_pln": t2_revenue_total.astype(int),
    "delta_pln": delta.astype(int),
    "delta_pct": np.round(pct, 1)
}).reset_index(drop=True)
display(t_delta)

In [ ]:
colors = ["forestgreen" if v >= 0 else "crimson" for v in t_delta['delta_pln']]
plt.figure(figsize=(12,4))
plt.bar(t_delta["category"], t_delta["delta_pln"], color=colors)
plt.show()

In [ ]:
colors = ["forestgreen" if v >= 0 else "crimson" for v in t_delta['delta_pln']]
plt.figure(figsize=(12,4))
plt.bar(t_delta["category"], t_delta["delta_pln"], color=colors)
plt.axhline(0, color="black", linewidth=1)
plt.show()

In [ ]:
def format_sep_signed(x: float) -> str:
    s = format_sep((abs(x)))
    return ("+" if x>=0 else "\u2212") + s

In [ ]:
def pad_ylims(axes, factor=0.07):
    ymin, ymax = axes.get_ylim() # pobranie wartości min i max dla y
    pad = (ymax - ymin) * factor
    plt.ylim(ymin - pad, ymax + pad) # ustawienie nowej wartości y max

In [ ]:
def label_bars(axes, bars, values, formatter=format_sep_signed, offset_factor=0.02):
    ymin, ymax = axes.get_ylim()
    offset = (ymax - ymin) * offset_factor
    for b, v in zip(bars, values):
        x = b.get_x() + b.get_width() / 2
        y = b.get_y() + b.get_height()
        label = formatter(v)
        va = "top" if v < 0 else "bottom"
        y_text = y-offset if v < 0 else y+offset
        axes.text(x, y_text, label, ha="center", va=va)

In [ ]:
plt.figure(figsize=(12,4))
bars = plt.bar(t_delta["category"], t_delta["delta_pln"], color=colors)
pad_ylims(plt.gca())
label_bars(plt.gca(), bars, t_delta["delta_pln"])
plt.axhline(0, color="black", linewidth=1)
plt.title("LUMA - Zmiana przychodu per kategoria w tygodniu 2 vs. 1")
plt.xlabel("Kategoria")
plt.ylabel("\u2206 Przychód (PLN)")
plt.tight_layout()
plt.show()

# Analiza i wizualizacja danyh- prezentacja zmian względnych (procentowych)

In [ ]:
t_delta["delta_pct_plot"] = np.nan_to_num(t_delta["delta_pct"], nan=0.0)

In [ ]:
def format_pct_signed(p: float) -> str:
    if np.isnan(p):
        return "n/a"
    if p == 0:
        return "0"
    s = format_sep(abs(p))
    return("+" if p>0 else "\u2212") + s + "%"

In [ ]:
colors_pct = ["green" if p > 0 else "crimson" if p < 0 else "silver" for p in t_delta["delta_pct_plot"]]
plt.figure(figsize=(12,4))
bars = plt.bar(t_delta["category"], t_delta["delta_pct_plot"], color=colors_pct)
pad_ylims(plt.gca())
label_bars(plt.gca(), bars, t_delta["delta_pct_plot"], formatter=format_pct_signed)
legend_patches = [
    Patch(facecolor="green", label = "wzrost"),
    Patch(facecolor="red", label = "spadek"),
    Patch(facecolor="silver", label = "0 lub n/a")
]
plt.axhline(0, color="black", linewidth=1)
plt.title("LUMA - Zmiana przychodu per kategoria w tygodniu 2 vs. 1")
plt.xlabel("Kategoria")
plt.ylabel("\u2206 Przychód (PLN)")
plt.tight_layout()
plt.show()



In [ ]:
t_delta

Indywidualna modyfikacja artystów serii danych

In [ ]:

plt.figure(figsize=(12,4))
bars = plt.bar(t_delta["category"], t_delta["delta_pct_plot"], color=colors_pct)
for b, p in zip(bars, t_delta["delta_pct_plot"]):
    if np.isnan(p) or p== 0:
        b.set_y(-3)
        b.set_height(6)
        b.set_color("silver")
pad_ylims(plt.gca())
label_bars(plt.gca(), bars, t_delta["delta_pct_plot"], formatter=format_pct_signed)
legend_patches = [
    Patch(facecolor="green", label = "wzrost"),
    Patch(facecolor="red", label = "spadek"),
    Patch(facecolor="silver", label = "0 lub n/a")
]
plt.axhline(0, color="black", linewidth=1)
plt.title("LUMA - Zmiana przychodu per kategoria w tygodniu 2 vs. 1")
plt.xlabel("Kategoria")
plt.ylabel("\u2206 Przychód (PLN)")
plt.tight_layout()
plt.show()



In [ ]:
def modify_zero_bars(bars, values, **kwargs):
    for b, v in zip(bars, values):
        if np.isnan(v) or v==0:
            b.set(**kwargs)

In [ ]:
def add_manual_legend(*kwargs):
    legends_patches = [Patch(**k) for k in kwarg]
    plt.legend(handles=legend_patches)
    

In [ ]:

plt.figure(figsize=(12,4))
bars = plt.bar(t_delta["category"], t_delta["delta_pct_plot"], color=colors_pct)
modify_zero_bars(bars, t_delta["delta_pct_plot"], y=-3, height=6, hatch="///", facecolor="none")
pad_ylines(plt.gca())
pad_ylines(plt.gca(), bars, t_delta["delta_pct_plot"], formatter=format_pct_signed)
add_manual_legend(
    {"facecolor":"green", "label":"wzrost"},
    {"facecolor":"red", "label":"spadek"},
    {"facecolor":"green", "label":"0 lub n/a"}
)

plt.axhline(0, color="black", linewidth=1)
plt.title("LUMA - Zmiana przychodu per kategoria w tygodniu 2 vs. 1")
plt.xlabel("Kategoria")
plt.ylabel("\u2206 Przychód (PLN)")
plt.tight_layout()
plt.show()



# Zmiany względne i bezwzględne na jednym wykresie

Pomocnicze współrzędne poziome słópków

In [ ]:
x_locks = np.arange(len(order))
bar_width = 0.45
bar_half = bar_width / 2

In [ ]:
fig = plt.figure(figsize=(12,4))
ax_pln = plt.gca()
ax_pct = ax_pln.twinx()
bars_pln = ax_pln.bar(x_locks-bar_half, t_delta["delta_pln"], width=bar_width, label="Tydzien 1", color=colors)
bars_pct = ax_pct.bar(x_locks+bar_half, t_delta["delta_pct_plot"], width=bar_width, label="Tydzien 2", color=colors_pct, alpha = 0.45)

In [ ]:
def align_value_expand(ax_ref, v_ref, ax_other, v_other, eps=1e-12):
    r0, r1 = ax_ref.get_ylim()
    if r0 == r1:
        return
    f = (v_ref - r0) / (r1-r0)

    y0, y1 = ax_other.get_ylim()
    inv = y1 < y0
    a0, a1 = (y1, y0) if inv else (y0, y1)
    if a1 == a0:
        return
    g = (v_other - a0) / (1-f)
    f = min(max(f, eps), 1-eps)

    if g < f - eps:
        a0p = (v_other - f*a1) / (a1 - f)
        a0_new, a1_new = min(a0, a0p), a1
    elif g > f + eps:
        a1p = (v_other - f*a1) / (1-f)
        a0_new, a1_new = min(a0, a0p), a1
    else:
        a0_new, a1_new = a0, a1
    ax_other.set_ylim(a1_new, a0_new) if inv else (a0_new, a1_new)

In [ ]:
fig = plt.figure(figsize=(12,4))
ax_pln = plt.gca()
ax_pct = ax_pln.twinx()
bars_pln = ax_pln.bar(x_locks-bar_half, t_delta["delta_pln"], width=bar_width, label="Tydzien 1", color=colors)
bars_pct = ax_pct.bar(x_locks+bar_half, t_delta["delta_pct_plot"], width=bar_width, label="Tydzien 2", color=colors_pct, alpha = 0.65)
align_value_expand(ax_pln, 0, ax_pct, 0)
ax_pct.axline(0, color="black", linewidth=1)
plt.show()

In [ ]:
t_delta["delta_pln_plot"] = [0 if np.abs(v) < 100 else v for v in t_delta["delta_pln"]]

In [ ]:
fig = plt.figure(figsize=(12,4))
ax_pln = plt.gca()
ax_pct = ax_pln.twinx()
bars_pln = ax_pln.bar(x_locks-bar_half, t_delta["delta_pln"], width=bar_width, label="Tydzien 1", color=colors)
bars_pct = ax_pct.bar(x_locks+bar_half, t_delta["delta_pct_plot"], width=bar_width, label="Tydzien 2", color=colors_pct, alpha = 0.65)
align_value_expand(ax_pln, 0, ax_pct, 0)
ax_pct.axline(0, color="black", linewidth=1)
plt.show()

modify_zero_bars(bars_pln, t_delta["delta_pln_plot"], y=-3, height=6, hatch="///", facecolor="none")
modify_zero_bars(bars_pln, t_delta["delta_pln_plot"], y=-3, height=6, hatch="///", facecolor="none")

pad_ylim(ax_pln)
pad_ylim(ax_pct)

label_bars(plt.gca(), bars, t_delta["delta_pct_plot"])
label_bars(plt.gca(), bars, t_delta["delta_pct_plot"], formatter=format_pct_signed)

align_value_expand(ax_pln, 0, ax_pct, 0)

add_manual_legend(
    {"facecolor":"green", "label":"wzrost"},
    {"facecolor":"red", "label":"spadek"},
    {"facecolor":"green", "label":"0 lub n/a", "hatch": "/////"}
)
plt.axhline(0, color="black", linewidth=1)
for x in x_locs[:-1] + 0.5:
    ax_pct.axvline(x, color="black", linewidth=0.5, alpha=0.3)
ax_pct.set_xticks(x_locks, order)
ax_pct.set_title("Luma - zmiana przychodu per kategoria w tyg 2 vs 1")
ax_pct.set_xlabel("Kategoria")
ax_pct.set_ylabel("Zmiana %")
ax_pln.set_ylabel("zmiana pln")